In [ ]:
# jax 学习

import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["NVIDIA_VISIBLE_DEVICES"] = "1"
import jax
import jax.numpy as jnp
# JAX 的 API 和 NumPy 几乎一模一样，但由于 JAX 的无状态特性，
# 它的随机数生成机制不同。你必须显式地创建一个 PRNGKey 并传递它。
key = jax.random.PRNGKey(42)

# 使用 key 生成一个包含 5 个正态分布随机数的数组
x = jax.random.normal(key, (5,))
print(f"生成的随机数组: \n{x}\n")

# JAX 可以对纯函数（没有副作用的函数）进行自动求导。
def f(w):
    # 一个简单的二次函数: f(w) = 2 * w^2 + 3 * w + 1
    return 2 * w**2 + 3 * w + 1

# 生成梯度
df_dw=jax.grad(f)
w_val = 2.0
print(f"f({w_val}) 的值: {f(w_val)}")
print(f"f({w_val}) 的导数值 (4*2+3): {df_dw(w_val)}\n")

print("=== 3. jax.jit: 即时编译加速 ===")
# JAX 使用 XLA (加速线性代数) 将纯 Python 代码编译成高效的机器码。
# 尤其在深度学习的循环中，加上 @jax.jit 装饰器可以让代码跑得飞快。

@jax.jit
def selu(x,alpha=1.67326,scale=1.0507):
    return scale * jax.numpy.where(x>0,x,alpha*(jnp.exp(x)-1))
# 第一次运行时，JAX 会先“编译”这个函数，因此稍微慢一点
print(f"SELU(x) 初次运行 (包含编译): \n{selu(x)}")
# 第二次运行时，直接调用编译好的代码，速度极快
print(f"SELU(x) 再次运行 (极速): \n{selu(x)}\n")


print("=== 4. jax.vmap: 自动向量化 ===")
# vmap 可以将一个处理“单个数据”的函数，自动升级为处理“批量数据”的函数，
# 从而避免在 Python 中写缓慢的 for 循环。

# 定义一个只处理单个元素的简单函数
@jax.jit
def calculate_discount(price, discount_rate):
    return price * (1.0 - discount_rate)

# 假设我们有一批价格和折扣率
prices = jnp.array([100.0, 250.0, 50.0, 10.0, 500.0])
rates = jnp.array([0.1, 0.2, 0.05, 0.0, 0.5])

# 使用 jax.vmap 自动将其向量化，无需重写函数
v_calculate = jax.vmap(calculate_discount)

discounted_prices = v_calculate(prices, rates)
print(f"原始价格: {prices}")
print(f"对应折扣: {rates}")
print(f"折后价格 (通过 vmap 批量计算): {discounted_prices}")

In [ ]:
# 得会用seaborn画图
import wandb
from pprint import pprint
ENTITY = "phil_ning" 
PROJECT_NAME = "aubo-i10-fintune"

api = wandb.Api()
path = f"{ENTITY}/{PROJECT_NAME}"
run=api.runs(path)[len(api.runs(path))-1]
pprint(run.state)
data = []

metric_keys=['Step',"train/lr","train/loss"]

config=run.config
pprint(config)



In [ ]:
import wandb
import numpy as np
import time
import math

# 配置 W&B 项目名称
PROJECT_NAME = "robot-rl-seaborn-demo"

def simulate_training(algorithm, seed, total_steps=100000, log_interval=5000):
    """
    模拟强化学习训练过程，生成带有一些噪声的曲线数据。
    """
    # 初始化 W&B Run
    run_name = f"{algorithm}_seed{seed}"
    wandb.init(
        project=PROJECT_NAME,
        name=run_name,
        group=algorithm, # 将相同算法的 runs 分组
        config={
            "algorithm": algorithm,
            "seed": seed,
            "learning_rate": 3e-4,
            "env": "PushT-v2",
            "total_steps": total_steps
        }
    )

    print(f"🚀 开始模拟训练: {run_name}")

    # 模拟不同的算法性能特征
    if algorithm == "PPO":
        base_success = 0.8
        growth_rate = 0.00005
        noise_level = 0.05
    elif algorithm == "SAC":
        base_success = 0.95
        growth_rate = 0.00008
        noise_level = 0.02
    else: # TD3
        base_success = 0.85
        growth_rate = 0.00006
        noise_level = 0.08

    for step in range(0, total_steps + 1, log_interval):
        # 模拟生成 success_rate (带有对数增长和随机噪声)
        # 使用 math.log1p 避免 log(0)
        progress = step * growth_rate
        raw_success = base_success * (1 - math.exp(-progress))
        
        # 添加随机噪声
        noise = np.random.normal(0, noise_level)
        success_rate = np.clip(raw_success + noise, 0.0, 1.0)

        # 模拟 episode_reward
        episode_reward = success_rate * 100 + np.random.normal(0, 5)

        # 🚨 核心：将指标记录到 W&B
        # 注意：我们将 env_step 作为普通的 metric 记录，而不是依赖 W&B 的内置 step
        wandb.log({
            "env_step": step,
            "eval/success_rate": success_rate,
            "eval/episode_reward": episode_reward,
            "train/loss": max(0, 1.0 - progress + np.random.normal(0, 0.1))
        })
        
        # 稍微暂停一下，模拟训练耗时
        time.sleep(0.01)

    print(f"✅ 完成模拟训练: {run_name}\n")
    wandb.finish()

if __name__ == "__main__":
    algorithms = ["PPO", "SAC", "TD3"]
    seeds = [0, 1, 2]

    print(f"开始向 W&B 项目 '{PROJECT_NAME}' 写入模拟数据...")
    for algo in algorithms:
        for seed in seeds:
            simulate_training(algo, seed)
    print("🎉 所有模拟数据已成功记录到 W&B！")


In [ ]:
import matplotlib.pyplot as plt
from pprint import pprint
# 打印所有可用的 rc 参数及其当前默认值
pprint(plt.rcParams)

In [ ]:
import wandb
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

# 必须与写入时的项目名称一致
ENTITY = "phil_ning" 
PROJECT_NAME = "robot-rl-seaborn-demo"

def fetch_data_from_wandb():
    """
    使用 W&B API 拉取指定项目中的所有运行数据，并合并为一个 DataFrame。
    """
    print(f"📥 正在从 W&B 拉取项目 '{PROJECT_NAME}' 的数据...")
    api = wandb.Api()
    
    # 构建项目路径
    path = f"{ENTITY}/{PROJECT_NAME}" if ENTITY else PROJECT_NAME
    runs = api.runs(path)

    all_data = []
    
    metric_keys = ["env_step", "eval/success_rate", "eval/episode_reward", "train/loss"]

    for run in runs:
        # 跳过未完成或失败的 runs
        if run.state != "finished":
            continue

        print(f"  - 拉取 Run: {run.name}")
        
        # 提取超参数配置
        config = run.config
        algorithm = config.get("algorithm", "Unknown")
        seed = config.get("seed", "Unknown")

        # 使用 scan_history 确保拉取到所有数据点
        rows = []
        for row in run.scan_history(keys=metric_keys):
            rows.append(row)
            
        if not rows:
            print(f"    ⚠️ Run {run.name} 没有包含所需指标的数据，已跳过。")
            continue

        # 转换为 DataFrame 并添加元数据
        df_run = pd.DataFrame(rows)
        df_run["algorithm"] = algorithm
        df_run["seed"] = seed
        df_run["run_id"] = run.id
        
        all_data.append(df_run)

    if not all_data:
        raise ValueError("没有拉取到任何有效数据，请检查项目名称或运行状态。")

    # 合并所有 runs 的数据
    df_combined = pd.concat(all_data, ignore_index=True)
    
    # 清理数据：丢弃包含 NaN 的行
    df_combined = df_combined.dropna(subset=metric_keys)
    
    print(f"✅ 数据拉取完成，共提取 {len(df_combined)} 条记录。")
    return df_combined

def plot_with_seaborn(df, context, style, palette):
    """
    使用 Seaborn 绘制高质量的训练曲线。
    """
    print(f"🎨 正在使用 Seaborn 绘制图表... (context={context}, style={style}, palette={palette})")
    
    try:
        sns.set_theme(
            context=context,        
            style=style,      
            palette=palette,
            font_scale=1.2,         
            rc={"lines.linewidth": 2.0} 
        )
    except Exception as e:
        print(f"⚠️ set_theme失败: {e}，将跳过该组合。")
        return

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 允许palette为字符串或dict, 如果是None就自动选择
    # 用dict时仍保证三类颜色一一对应
    if isinstance(palette, dict):
        pal_for_plot = palette
    else:
        pal_for_plot = None

    # --- 绘制图表 1: Success Rate ---
    ax1 = axes[0]
    sns.lineplot(
        data=df, x="env_step", y="eval/success_rate", hue="algorithm",
        palette=pal_for_plot, errorbar=("ci", 95), estimator="mean", ax=ax1
    )
    ax1.set_title("Evaluation Success Rate over Time", pad=15, fontweight='bold')
    ax1.set_xlabel("Environment Steps")
    ax1.set_ylabel("Success Rate")
    ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, pos: f"{int(x/1000)}k" if x >= 1000 else f"{int(x)}"))
    handles, labels = ax1.get_legend_handles_labels()
    ax1.legend(handles=handles, labels=labels, title="Algorithm", frameon=False, loc="lower right")

    # --- 绘制图表 2: Episode Reward ---
    ax2 = axes[1]
    sns.lineplot(
        data=df, x="env_step", y="eval/episode_reward", hue="algorithm",
        palette=pal_for_plot, errorbar=("ci", 95), estimator="mean", ax=ax2
    )
    ax2.set_title("Episode Reward over Time", pad=15, fontweight='bold')
    ax2.set_xlabel("Environment Steps")
    ax2.set_ylabel("Reward")
    ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, pos: f"{int(x/1000)}k" if x >= 1000 else f"{int(x)}"))
    ax2.get_legend().remove()

    # --- 绘制图表 3: Training Loss ---
    ax3 = axes[2]
    sns.lineplot(
        data=df, x="env_step", y="train/loss", hue="algorithm",
        palette=pal_for_plot, errorbar=("ci", 95), estimator="mean", ax=ax3
    )
    ax3.set_title("Training Loss over Time", pad=15, fontweight='bold')
    ax3.set_xlabel("Environment Steps")
    ax3.set_ylabel("Loss")
    ax3.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, pos: f"{int(x/1000)}k" if x >= 1000 else f"{int(x)}"))
    ax3.get_legend().remove()

    sns.despine(fig)
    plt.tight_layout()
    plt.show()
    

if __name__ == "__main__":
    # 1. 获取数据
    df_metrics = fetch_data_from_wandb()
    
    # 遍历所有 theme 组合（包括 style, context, palette）
    print("🌀 遍历所有Seaborn样式(context, style, palette)...")
    import itertools

    # seaborn官方的主题style
    styles = ['white', 'dark', 'whitegrid', 'darkgrid', 'ticks'] 
    # seaborn官方的 context
    contexts = ['paper', 'notebook', 'talk', 'poster']
    # seaborn官方调色板
    # 获取所有可用调色板名称(排除以"_r"结尾和"husl/Set1等易混乱的)，限制数量避免图多
    palette_names = [
        "deep", "muted", "pastel", "bright", "dark", "colorblind", 
        "tab10", "Set1", "Set2", "Set3", "Paired", "Accent"
    ]
    # 可选手动palette，也可用None, 实际使用palette名自动适配类别
    custom_palettes = [None]

    # 再加上常用自定义dict
    custom_palettes.append({"PPO": "#4C72B0", "SAC": "#DD8452", "TD3": "#55A868"})

    pal_source = palette_names + custom_palettes

    combinations = itertools.product(contexts, styles, pal_source)

    for context, style, palette in combinations:
        # palette可以是名字或dict或None
        print(f"Plotting with context='{context}', style='{style}', palette='{palette}'")
        try:
            plot_with_seaborn(df_metrics, context, style, palette)
        except Exception as e:
            print(f"Failed for context={context}, style={style}, palette={palette}: {e}")


In [ ]:

import plotly.graph_objects as go
import numpy as np

fig = go.Figure()
d_model = 512
x = np.arange(d_model)
positions = np.arange(0, 10)  # 展示从0到9所有位置的编码分子
for pos in positions:
    numerator = pos * np.exp(x / d_model)
    fig.add_trace(go.Scatter(x=x, y=numerator, mode='lines', name=f'pos={pos}'))

fig.update_layout(
    title='Positional Encoding Numerator for Multiple Positions',
    xaxis_title='Dimension',
    yaxis_title='Numerator Value',
    legend_title='Position'
)
fig.show()


In [ ]:
import sys
import random
import numpy as np
import os
from PIL import Image
from core.MyEnv import MyEnv
from lerobot.datasets.lerobot_dataset import LeRobotDataset

In [ ]:
# Layout randomness: keep SEED=None and call `reset()` with no args between episodes
# so NumPy's RNG advances and cube pose / target position change each time.
# Pass an int to MyEnv(..., seed=K) only when you need a reproducible *first* scene;
# do not pass seed into every `reset()` during collection, or you repeat the same layout.


REPO_NAME = 'ningyv/auboI10'
NUM_DEMO = 10 # Number of demonstrations to collect
ROOT = "/Users/ningyu/code_before_paper/MyI10Tele/data2" # The root directory to save the demonstrations

In [ ]:
I10_path = '/Users/ningyu/code_before_paper/MyI10Tele/assets/aubo_i10_2/aubo_i10.xml'
import mujoco 
model = mujoco.MjModel.from_xml_path(I10_path)
print(model.body_pos)

In [ ]:
TASK_NAME = 'Put cube on the black platform' 
xml_path = '/Users/ningyu/code_before_paper/MyI10Tele/assets/aubo_i10_inspire/myscene.xml'
# xml_path = './asset/example_scene_y_i10.xml'
# Define the environment
PnPEnv = MyEnv(xml_path, seed=42)
print(f"action_type: {PnPEnv.action_type}")
print(f"state_type: {PnPEnv.state_type}")

In [ ]:
create_new = True
if os.path.exists(ROOT):
    print(f"Directory {ROOT} already exists.")
    ans = input("Do you want to delete it? (y/n) ")
    if ans == 'y':
        import shutil
        shutil.rmtree(ROOT)
    else:
        create_new = False


if create_new:
    dataset = LeRobotDataset.create(
                repo_id=REPO_NAME,
                root = ROOT, 
                robot_type="aubo_i10_inspire",
                fps=20, # 20 frames per second
                features={
                    "observation.image": {
                        "dtype": "video",
                        "shape": (256, 256, 3),
                        "names": ["height", "width", "channels"],
                    },
                    "observation.wrist_image": {
                        "dtype": "video",
                        "shape": (256, 256, 3),
                        "names": ["height", "width", "channel"],
                    },
                    "observation.state": {
                        "dtype": "float32",
                        # "shape": (7 if PnPEnv.state_type == 'qpos' else 6,),
                        "shape": (7 ,),
                        "names": ["state"], # 6 joint angles and 1 gripper ////  x, y, z, roll, pitch, yaw
                    },
                    "action": {
                        "dtype": "float32",
                        # "shape": (6 if PnPEnv.action_type == 'ee_pose' else 7,),
                        "shape": (7,),
                        "names": ["action"], # x, y, z, roll, pitch, yaw /// 6 joint angles and 1 gripper
                    },
                    "obj_init": {
                        "dtype": "float32",
                        "shape": (6,),
                        "names": ["obj_init"], # just the initial position of the object. Not used in training.
                    },
                },
                image_writer_threads=10,
                image_writer_processes=5,
        )
else:
    print("Load from previous dataset")
    dataset = LeRobotDataset(REPO_NAME, root=ROOT)

In [ ]:
action = np.zeros(7)
episode_id = 0
record_flag = False # Start recording when the robot starts moving
try:
    while PnPEnv.env.is_viewer_alive() and episode_id < NUM_DEMO:
        PnPEnv.step_env()
        if PnPEnv.env.loop_every(HZ=20):
            # check if the episode is done
            done = PnPEnv.check_success()
            if done: 
                # Save the episode data and reset the environment
                dataset.save_episode()
                PnPEnv.reset()
                episode_id += 1
                record_flag = False
            # Teleoperate the robot and get delta end-effector pose with gripper
            action, reset  = PnPEnv.teleop_robot()
            if not record_flag and sum(action) != 0:
                record_flag = True
                print("Start recording")
            if reset:
                # Reset the environment and clear the episode buffer
                # This can be done by pressing 'z' key
                PnPEnv.reset()
                dataset.clear_episode_buffer()
                record_flag = False
            # Step the environment
            # Get the end-effector pose and images
            # obs_action = PnPEnv.get_ee_pose()
            obs_action=PnPEnv.get_obs_action()
            # assert obs_action.type == PnPEnv.action_type , print(f"expect action_type: {PnPEnv.action_type}, but got {obs_action.type}")
            assert obs_action.type == "qpos"
            agent_image,wrist_image = PnPEnv.grab_image()
            # # resize to 256x256
            agent_image = Image.fromarray(agent_image)
            wrist_image = Image.fromarray(wrist_image)
            agent_image = agent_image.resize((256, 256))
            wrist_image = wrist_image.resize((256, 256))
            agent_image = np.array(agent_image)
            wrist_image = np.array(wrist_image)
            obs_state = PnPEnv.step(action)
    
            # from IPython.display import display, clear_output
            # clear_output(wait=True)
            # print(f"gripper_qpos: {PnPEnv.env.get_qpos_joint('rh_r1')}") # close : 0.81454458 open :2.7e-6
            # print(f"gripper_qpos: {PnPEnv.env.get_qpos_joint('rh_r1')[0]}")
            
            assert obs_state.type == PnPEnv.state_type, f"expect state_type: {PnPEnv.state_type}, but got {obs_state.type}"
            if record_flag:
                # Add the frame to the dataset
                dataset.add_frame({
                    "observation.image": agent_image,
                    "observation.wrist_image": wrist_image,
                    "observation.state": obs_state,
                    "action": obs_action,
                    "obj_init": PnPEnv.obj_init_pose,
                    "task": TASK_NAME,
                })
                # print(PnPEnv.obj_init_pose)
                print(f"cube pos {PnPEnv.env.get_p_body('cube')}")
                print(f"target pos {PnPEnv.env.get_p_body('place_target_platform')}")
            PnPEnv.render(teleop=True)

except Exception as e:
    print(f"Interrupted: {e}")
finally:
    PnPEnv.env.close_viewer()
    dataset.stop_image_writer()
    dataset.finalize()